# Exploratory Data Analysis Before Cleaning

Inspect the full `merged_order_items` dataset from `data/interim` before deciding how to clean it.
Each row represents one order item, identified by `(order_id, order_item_id)`; order attributes can repeat across items.

- Use only the full merged dataset through the existing schema-aware loader. Do not reload raw tables or rerun merging.
- Keep all statuses, missing values and unusual records for inspection. Do not filter, fill, normalize or overwrite the input.
- Calculate diagnostic summaries in separate objects. Use the appropriate item, order or entity denominator for each question.
- Display tables and plots in this notebook. Do not export datasets or generate/update documentation.
- Business feature engineering, preprocessing and delivery-performance analysis belong to later work.


## Project setup

Run the notebook from the repository root or a directory inside the repository.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


# Reuse the project-root discovery convention from the merge notebook.
def find_project_root(start_path: Path) -> Path:
    """Find the project root containing src/ and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path

    raise FileNotFoundError(
        "Could not locate project root containing both 'src/' and 'requirements.txt'."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from src.data_loader import load_data
from src.paths import INTERIM_DIR
from src.pipeline.contracts.schema import BENCHMARK_ROW_COUNT, FULL_MERGE_SCHEMA

print(f"Project root: {PROJECT_ROOT}")
print(f"Interim data path: {INTERIM_DIR}")


## Load the full input

The default reader tries Parquet first, then falls back to CSV if needed, restoring the registered schema.
To select CSV directly, add `file_format="csv"` to the call below.

Keep `input_baseline` in memory for the final non-mutation check. Its row hashes include the index; columns and dtypes are recorded separately.
No baseline file or run metadata is written.


In [ ]:
df = load_data("merged_order_items", stage="interim")

# Record the loaded input without making a second copy of the full dataset.
input_baseline = {
    "shape": df.shape,
    "columns": df.columns.copy(),
    "dtypes": df.dtypes.copy(),
    "row_hashes": pd.util.hash_pandas_object(df, index=True).copy(),
}


## Input baseline

Reference counts describe the current Olist full merge, rather than a cleaning rule.
Show observed counts beside them; investigate differences without modifying the input to match.
The loader already validates the registered schema. A unique-order count does not change the item-level grain of `df`.


In [ ]:
baseline_summary = pd.DataFrame(
    [
        ["Item rows", BENCHMARK_ROW_COUNT, len(df)],
        ["Columns", len(FULL_MERGE_SCHEMA.columns), len(df.columns)],
        ["Unique orders", 98_666, df["order_id"].nunique()],
    ],
    columns=["Metric", "Reference", "Observed"],
)
baseline_summary["Matches reference"] = baseline_summary["Reference"].eq(
    baseline_summary["Observed"]
)
display(baseline_summary)

if not baseline_summary["Matches reference"].all():
    print("Reference counts differ. Check the input version before interpreting later profiles.")

print("Input baseline retained in memory for the final non-mutation check.")


## Initial inspection

Preview the loaded table and its existing pandas summaries. These summaries are preliminary;
subsequent profiling will distinguish identifiers, categories, timestamps and measurements at the appropriate grain.


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")